<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/week_8_capstone_project/week8_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 capstone: an Elden Ring build and lore assistant

**The problem.** Players ask two kinds of questions about Elden Ring. Some are lookups ("where is
this boss, what does it drop?"). Others need a check or some arithmetic ("can I wield this weapon
with 20 Str and 12 Dex?", "how much heavier is this greatsword than that axe?"). A plain chatbot
guesses at both from memory and gets stats wrong with confidence. This assistant answers only from
game data pulled from the open Elden Ring Fan API, uses tools for anything that needs a check or a
calculation, and says "I don't have that" when the data does not cover the question.

**Four techniques, each reused from an earlier week:**

| Week | Technique | What it does here | Changed for this project |
|---|---|---|---|
| 5 | Retrieval (MiniLM embeddings + FAISS) | finds the records relevant to a question | one chunk per game record instead of fixed-size text chunks |
| 4 | Tool use (schemas, validator, `dispatch`, guarded code-runner, Claude tool loop) | exact stat lookups, requirement checks, arithmetic | the guarded runner and dispatcher are spliced in unchanged; new tools `get_weapon_stats`, `check_requirements` |
| 7 | Retrieval-selected dynamic few-shot | the **router**: decides `lookup` or `tool` for each question from its 4 most similar labeled questions | the same method, applied to routing instead of ticket labels |
| 3 | Grounded prompt | answer only from the context and tool results, decline otherwise | judged below for grounding |

**Architecture.**

```
question
   |
   v
[Week 7 router: 4 nearest labeled questions -> Claude Haiku]
   |                                   |
 "lookup"                            "tool"
   |                                   |
   v                                   v
[Week 5 retrieve top-5 records]   [Week 5 retrieve top-5 records]
   |                                   |
   v                                   v
[Week 3 grounded prompt]          [Week 4 tool loop: get_weapon_stats,
   |                                 check_requirements, calculate (guarded)]
   v                                   v
 answer ---------------------------> answer
                   |
                   v
   [LLM-as-judge, a different Claude model: correctness, grounding, calibration]
```

**Models.** Claude Haiku (`claude-haiku-4-5-20251001`) generates, routes and runs the tool loop. The
starter wired Gemini; this is the same swap disclosed in Weeks 3-7. The **judge is a different Claude
model** (`JUDGE_MODEL` below) so the system is not grading its own writing; it is still the same model
family, which is a limit I come back to.

**How to read the numbers.** The sandbox this notebook was built in cannot reach the Fan API or the
Anthropic API. The cells were checked against fakes first, which only proves the plumbing. Every number
and claim about behavior comes from the run in Colab. Anything written after seeing results is labeled.

**Data.** [Elden Ring Fan API](https://docs.eldenring.fanapis.com/): community data scraped from fan
sites, not affiliated with FromSoftware, free to use "fairly and not spam it". The notebook downloads it
once into `data/raw/` and reuses the cache. Do not commit `data/raw/`; the data is not mine to
redistribute and the notebook re-fetches it in a few calls.

In [ ]:
%pip install -q sentence-transformers faiss-cpu anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 8.4 MB/s eta 0:00:00


## Setup: keys, models, and a smoke test

Same Colab-safe key lookup as Weeks 5-7. Each model gets one tiny test call before anything else runs,
so a bad key or a model name your key cannot use fails here in seconds, not halfway through the eval.

In [ ]:
import os, re, json, math, time, ast, hashlib, pathlib, random, getpass, difflib
import urllib.request, urllib.error
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError
import numpy as np

os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
SEED = 8
K = 5                                                # records retrieved per question
GEN_MODEL = 'claude-haiku-4-5-20251001'              # generator, router and tool loop (Weeks 3-7)
JUDGE_MODEL = 'claude-sonnet-4-5-20250929'           # a DIFFERENT Claude model; edit this line if your key cannot use it

def _get_api_key():
    """Colab secret, then env var, then an in-notebook prompt (same pattern as Weeks 5-7)."""
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY')
    if key:
        return key
    try:
        return getpass.getpass('Enter your ANTHROPIC_API_KEY: ')
    except Exception:
        return None

_API_KEY = _get_api_key()
_client = None
if _API_KEY:
    import anthropic
    _client = anthropic.Anthropic(api_key=_API_KEY, timeout=30.0, max_retries=1)   # fail fast instead of hanging

TEMP_PINNED = True      # flips to False (with a printed note) if the API refuses temperature for a model

def create(model, retries=3, **kw):
    """One place for every model call. -> (response, error). Temperature is pinned to 0 through
    extra_body because it is not a named argument in every SDK version (the Week 7 bug)."""
    global TEMP_PINNED
    last = None
    for attempt in range(retries):
        try:
            args = dict(kw)
            if TEMP_PINNED:
                args['extra_body'] = {'temperature': 0}
            return _client.messages.create(model=model, **args), None
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
            if TEMP_PINNED and getattr(e, 'status_code', None) == 400 and 'temperature' in str(e).lower():
                TEMP_PINNED = False
                print('NOTE: the API refused temperature for this model -- continuing WITHOUT pinning it. '
                      'Outputs may vary slightly between runs; say so in the write-up.')
                continue
            if getattr(e, 'status_code', None) in (400, 401, 403, 404):
                break                     # bad key / model name / request: retrying cannot help
            time.sleep(2 ** attempt)
    return None, last

def probe(model):
    if _client is None:
        return False, 'no API key'
    r, err = create(model, retries=2, max_tokens=8, messages=[{'role': 'user', 'content': 'Reply with the single word: ok'}])
    return r is not None, err

GEN_OK, gen_err = probe(GEN_MODEL)
JUDGE_OK, judge_err = probe(JUDGE_MODEL) if GEN_OK else (False, 'skipped (generator unavailable)')
API_OK = GEN_OK
print(f'generator {GEN_MODEL}: ' + ('ready' if GEN_OK else f'[API-BLOCKED] {gen_err}'))
print(f'judge     {JUDGE_MODEL}: ' + ('ready' if JUDGE_OK else f'[JUDGE-BLOCKED] {judge_err}  (edit JUDGE_MODEL above; the rest still runs)'))
if JUDGE_OK and JUDGE_MODEL == GEN_MODEL:
    print('WARNING: judge and generator are the same model; say so in the write-up.')

/tmp/ipykernel_3387/2803839108.py:48: DeprecationWarning: The model 'claude-sonnet-4-5-20250929' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  return _client.messages.create(model=model, **args), None


generator claude-haiku-4-5-20251001: ready
judge     claude-sonnet-4-5-20250929: ready


## Part 1: Integrate

### 1a. The data (and the Week 5 retrieval stack over it)

Weapons and bosses are required; talismans, incantations, sorceries and shields are added when the API
serves them. The fetch asks for everything in one request and falls back to paging if the server caps
`limit`. The raw responses are cached in `data/raw/`.

In [ ]:
BASE = 'https://eldenring.fanapis.com/api'
RAW_DIR = pathlib.Path('data/raw'); RAW_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED_EP = ['weapons', 'bosses']
OPTIONAL_EP = ['talismans', 'incantations', 'sorceries', 'shields']
REFRESH = False                      # True re-downloads even if data/raw/ already has the file

def _get_json(url, tries=3):
    last = None
    for a in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'cosc650-week8-capstone/1.0'})
            with urllib.request.urlopen(req, timeout=30) as r:
                return json.loads(r.read().decode('utf-8'))
        except urllib.error.HTTPError as e:
            if e.code in (400, 401, 403, 404):
                raise
            last = e; time.sleep(2 ** a)
        except Exception as e:
            last = e; time.sleep(2 ** a)
    raise last

def fetch_all(endpoint):
    cache = RAW_DIR / f'{endpoint}.json'
    if cache.exists() and not REFRESH:
        return json.loads(cache.read_text())
    head = _get_json(f'{BASE}/{endpoint}?limit=1')
    total = int(head.get('total') or len(head.get('data', [])))
    got = {}
    body = _get_json(f'{BASE}/{endpoint}?limit={max(total, 1)}')
    for r in body.get('data', []):
        got[r.get('id') or r.get('name')] = r
    if len(got) < total:                          # the server capped limit: page through instead
        size = max(len(body.get('data', [])), 1)
        empty_in_a_row = 0
        for page in range(0, 200):
            before = len(got)
            body = _get_json(f'{BASE}/{endpoint}?limit={size}&page={page}')
            for r in body.get('data', []):
                got[r.get('id') or r.get('name')] = r
            time.sleep(0.3)                       # the API asks to be used fairly
            empty_in_a_row = empty_in_a_row + 1 if len(got) == before else 0
            if len(got) >= total or empty_in_a_row >= 3:      # 1-indexed servers repeat page 1 for page=0 and page=1
                break
    if len(got) < total:
        print(f'  WARNING: {endpoint}: got {len(got)} of {total} records')
    records = list(got.values())
    cache.write_text(json.dumps(records))
    return records

RAW = {}
for ep in REQUIRED_EP:
    try:
        RAW[ep] = fetch_all(ep)
    except Exception as e:
        raise RuntimeError(f'Could not download the required endpoint {ep!r} from {BASE}: {type(e).__name__}: {e}. '
                           'The Fan API is a free community service; check your connection and try again later.')
for ep in OPTIONAL_EP:
    try:
        RAW[ep] = fetch_all(ep)
    except Exception as e:
        print(f'  optional endpoint {ep!r} skipped: {type(e).__name__}: {e}')
print({k: len(v) for k, v in RAW.items()})

{'weapons': 307, 'bosses': 106, 'talismans': 87, 'incantations': 98, 'sorceries': 71, 'shields': 69}


In [ ]:
# ---- turn each record into one retrievable chunk (one record = one chunk) ----
KIND = {'weapons': 'weapon', 'bosses': 'boss', 'talismans': 'talisman', 'incantations': 'incantation',
        'sorceries': 'sorcery', 'shields': 'shield'}
ATTR = {'str': 'Str', 'dex': 'Dex', 'int': 'Int', 'fai': 'Fai', 'arc': 'Arc'}

def attr_key(n):
    return ATTR.get(str(n).strip().lower()[:3])

def norm_name(s):
    s = re.sub(r'\s+', ' ', str(s)).strip().lower()
    return re.sub(r'^the ', '', s)

def num(x):
    if isinstance(x, bool):
        return None
    if isinstance(x, (int, float)):
        return x
    try:
        return float(str(x).replace(',', ''))
    except Exception:
        return None

def _pairs(lst, k1, k2):
    return ', '.join(f"{x.get(k1)} {x.get(k2)}" for x in (lst or []) if isinstance(x, dict))

def gear_text(kind, r):
    parts = [f"{kind.title()}: {r['name']}"]
    if r.get('category'):
        parts.append(f"Category: {r['category']}")
    if num(r.get('weight')) is not None:
        parts.append(f"Weight: {r['weight']}")
    parts.append(f"Requires: {_pairs(r.get('requiredAttributes'), 'name', 'amount') or 'none listed'}")
    sc = _pairs(r.get('scalesWith'), 'name', 'scaling')
    if sc:
        parts.append(f"Scales with: {sc}")
    atk = ', '.join(f"{a.get('name')} {a.get('amount')}" for a in (r.get('attack') or [])
                    if isinstance(a, dict) and num(a.get('amount')))
    if atk:
        parts.append(f"Attack: {atk}")
    parts.append(f"Description: {r.get('description', '')}")
    return ' | '.join(parts)

def boss_text(r):
    parts = [f"Boss: {r['name']}"]
    if r.get('region'):
        parts.append(f"Region: {r['region']}")
    if r.get('location'):
        parts.append(f"Location: {r['location']}")
    if r.get('drops'):
        parts.append("Drops: " + '; '.join(str(d) for d in r['drops']))
    if r.get('healthPoints'):
        parts.append(f"Health: {r['healthPoints']}")
    parts.append(f"Description: {r.get('description', '')}")
    return ' | '.join(parts)

def generic_text(kind, r):
    parts = [f"{kind.title()}: {r['name']}"]
    for k, v in r.items():
        if k in ('id', 'image', 'name'):
            continue
        if isinstance(v, (str, int, float)) and str(v).strip():
            parts.append(f"{k}: {v}")
        elif isinstance(v, list) and v and all(isinstance(x, (str, int, float)) for x in v):
            parts.append(f"{k}: " + '; '.join(str(x) for x in v))
        elif isinstance(v, list) and v and all(isinstance(x, dict) for x in v):
            parts.append(f"{k}: " + ', '.join(' '.join(str(y) for y in x.values()) for x in v))
    return ' | '.join(parts)

CHUNKS = []
for ep, recs in RAW.items():
    kind = KIND[ep]
    for r in recs:
        if not isinstance(r, dict) or not str(r.get('name', '')).strip():
            continue
        r['name'] = str(r['name']).strip()
        text = boss_text(r) if kind == 'boss' else (gear_text(kind, r) if kind in ('weapon', 'shield') else generic_text(kind, r))
        CHUNKS.append({'id': len(CHUNKS), 'kind': kind, 'name': r['name'], 'text': text, 'rec': r})

NAME_COUNT = Counter((c['kind'], norm_name(c['name'])) for c in CHUNKS)
BY_NAME = defaultdict(list)
for c in CHUNKS:
    BY_NAME[(c['kind'], norm_name(c['name']))].append(c['id'])
FINGERPRINT = hashlib.sha256('\n'.join(sorted(f"{c['kind']}:{c['name']}" for c in CHUNKS)).encode()).hexdigest()[:16]

print(f'{len(CHUNKS)} chunks:', dict(Counter(c['kind'] for c in CHUNKS)))
print('duplicate names (same kind):', sum(1 for v in NAME_COUNT.values() if v > 1))
print('dataset fingerprint:', FINGERPRINT, '(sorted kind:name list; compare it if the numbers ever differ between runs)')
print('\nsample chunk:\n ', next(c['text'] for c in CHUNKS if c['kind'] == 'weapon')[:400])
print('\nsample chunk:\n ', next(c['text'] for c in CHUNKS if c['kind'] == 'boss')[:400])

738 chunks: {'weapon': 307, 'boss': 106, 'talisman': 87, 'incantation': 98, 'sorcery': 71, 'shield': 69}
duplicate names (same kind): 3
dataset fingerprint: 7d41d82dd40e5dc1 (sorted kind:name list; compare it if the numbers ever differ between runs)

sample chunk:
  Weapon: Hand Axe | Category: Axe | Weight: 3.5 | Requires: Str 9, Dex 8 | Scales with: Str D, Dex D | Attack: Phy 113, Crit 100 | Description: Commonly known as a hatchet, this smaller variety of axe is an everyday work tool. Despite its short reach, it is easy to handle, exceling in successive attacks.

sample chunk:
  Boss: Abductor Virgins | Region: Mount Gelmir | Location: Volcano Manor | Drops: 10.000 Runes; Inquisitor's Girandole | Health: ??? | Description: Deadly mechanical constructs who wield massive blades.


In [ ]:
# ---- Week 5 retrieval: MiniLM embeddings + FAISS inner-product index (same stack, same model) ----
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
cvec = embedder.encode([c['text'] for c in CHUNKS], normalize_embeddings=True, batch_size=64,
                       show_progress_bar=False).astype('float32')
index = faiss.IndexFlatIP(cvec.shape[1]); index.add(cvec)

def retrieve(q, k=K):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    ids = index.search(qe, k)[1][0]
    return [int(i) for i in ids if i >= 0]          # FAISS uses -1 for missing results

print('index ready:', index.ntotal, 'chunks')
for i in retrieve('how heavy is the hand axe', 3):
    print(' ', CHUNKS[i]['text'][:110])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

index ready: 738 chunks
  Weapon: Battle Axe | Category: Axe | Weight: 4.5 | Requires: Str 12, Dex 8 | Scales with: Str D, Dex D | Attac
  Weapon: Warped Axe | Category: Axe | Weight: 7.5 | Requires: Str 24, Dex 8 | Scales with: Str C | Attack: Phy 
  Weapon: Longhaft Axe | Category: Greataxe | Weight: 12.5 | Requires: Str 24, Dex 8 | Scales with: Str ?, Dex ?


### 1b. Tools (Week 4)

The guarded code-runner (`calculate`), the argument validator and `dispatch` below are copied
**unchanged** from the Week 4 notebook: arithmetic only, an allowlist checked before evaluation, a
size cap on `**`, a time limit. Only the tools around them are new. `get_weapon_stats` returns the
record as structured numbers (so the model does not do arithmetic on text it read), and
`check_requirements` is the "can I wield it?" check. A weapon name must match exactly (ignoring case
and a leading "the"); near misses come back as suggestions, never as a silent guess.

In [ ]:
# ---- copied verbatim from the Week 4 notebook (guarded code-runner) ----
# ---------- guarded code-runner ----------
# PERMITS: arithmetic on numeric literals only -- + - * / ** unary minus, and parentheses.
# BLOCKS (explicitly, by construction): filesystem access, network access, process/subprocess
# execution, imports, attribute access, name/variable lookup, function calls of any kind,
# comprehensions, lambdas -- anything not in the allowlist is rejected before evaluation.
import ast, json, math, time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError

_ALLOWED_NODE_TYPES = (
    ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
    ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.USub, ast.UAdd,
)
_MAX_EXPR_LEN = 200
_TIME_LIMIT_SECONDS = 2.0
_MAX_RESULT_BITS = 256   # generous for real trip-budget dollar amounts, tiny next to a bignum blowup
_executor = ThreadPoolExecutor(max_workers=2)

class GuardrailViolation(Exception):
    pass

def _check_allowed(tree):
    """Pre-execution check: walk the whole parsed tree and reject anything outside
    the allowlist before any evaluation happens."""
    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODE_TYPES):
            raise GuardrailViolation(f'disallowed expression element: {type(node).__name__}')
        if isinstance(node, ast.Constant) and type(node.value) not in (int, float):
            raise GuardrailViolation(f'disallowed constant type: {type(node.value).__name__}')

_OPS = {ast.Add: lambda a,b: a+b, ast.Sub: lambda a,b: a-b, ast.Mult: lambda a,b: a*b,
        ast.Div: lambda a,b: a/b, ast.Pow: lambda a,b: a**b}
_UOPS = {ast.USub: lambda a: -a, ast.UAdd: lambda a: +a}

def _bit_length(x):
    return abs(x).bit_length() if isinstance(x, int) else 0

def _eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        left = _eval(node.left)
        right = _eval(node.right)
        op = type(node.op)
        if op is ast.Pow:
            # Estimate the result's bit-length WITHOUT computing it (base_bits * exponent
            # upper-bounds it), and reject before ever calling `left ** right`. This matters:
            # a huge pow() like 9**387420489 is what actually hangs -- checking the result's
            # size only after computing it is too late, the expensive call already ran.
            if isinstance(right, (int, float)) and abs(right) > 10_000:
                raise GuardrailViolation('exponent too large')
            if _bit_length(left) * abs(right) > _MAX_RESULT_BITS:
                raise GuardrailViolation(f'result of ** would exceed the {_MAX_RESULT_BITS}-bit size limit')
        result = _OPS[op](left, right)
        if _bit_length(result) > _MAX_RESULT_BITS:
            raise GuardrailViolation(f'intermediate result exceeds the {_MAX_RESULT_BITS}-bit size limit')
        return result
    if isinstance(node, ast.UnaryOp):
        return _UOPS[type(node.op)](_eval(node.operand))
    raise GuardrailViolation(f'unhandled node at eval time: {type(node).__name__}')

def _run_guarded(expression):
    if len(expression) > _MAX_EXPR_LEN:
        raise GuardrailViolation(f'expression too long ({len(expression)} chars, max {_MAX_EXPR_LEN})')
    tree = ast.parse(expression, mode='eval')
    _check_allowed(tree)   # pre-execution check
    return _eval(tree.body)

def calculate_trip_budget(expression):
    # NOTE on the time limit: a ThreadPoolExecutor + future.result(timeout=...) only stops the
    # *main* thread from waiting -- it can't forcibly kill the worker thread. Found this testing
    # against '9**9**9**9**9': it hung far past its time budget, because CPython's big-integer **
    # doesn't release the GIL mid-computation, so one huge pow() call blocks the whole interpreter
    # and can't be preempted by another thread's timeout. The real fix is the bit-length check in
    # _eval above, which rejects the exponentiation before the expensive pow() call ever runs. This
    # timeout stays as a second layer for anything else unexpectedly slow, but on its own it is not
    # a reliable defense against a single expensive built-in operation.
    future = _executor.submit(_run_guarded, expression)
    try:
        result = future.result(timeout=_TIME_LIMIT_SECONDS)   # time limit (secondary defense)
    except FutureTimeoutError:
        raise GuardrailViolation(f'execution exceeded {_TIME_LIMIT_SECONDS}s time limit')
    if not math.isfinite(result):
        raise GuardrailViolation('result is not a finite number')
    return result

In [ ]:
# ---- new tools, on top of the Week 4 machinery ----
WEAPON_BY_NAME = defaultdict(list)
for c in CHUNKS:
    if c['kind'] == 'weapon':
        WEAPON_BY_NAME[norm_name(c['name'])].append(c['rec'])

def weapon_req(rec):
    out = {}
    for a in (rec.get('requiredAttributes') or []):
        if isinstance(a, dict):
            k, v = attr_key(a.get('name')), num(a.get('amount'))
            if k and v is not None:
                out[k] = int(v) if float(v) == int(v) else v
    return out

def _find_weapon(name):
    recs = WEAPON_BY_NAME.get(norm_name(name))
    if not recs:
        close = difflib.get_close_matches(norm_name(name), list(WEAPON_BY_NAME), n=3, cutoff=0.75)
        return None, {'found': False, 'weapon': name, 'suggestions': [WEAPON_BY_NAME[c][0]['name'] for c in close]}
    if len(recs) > 1:
        return None, {'found': False, 'weapon': name, 'note': 'several weapons share this name'}
    return recs[0], None

def get_weapon_stats(weapon):
    r, err = _find_weapon(weapon)
    if err:
        return err
    return {'found': True, 'name': r['name'], 'category': r.get('category'), 'weight': num(r.get('weight')),
            'required': weapon_req(r),
            'scaling': {attr_key(s.get('name')) or s.get('name'): s.get('scaling')
                        for s in (r.get('scalesWith') or []) if isinstance(s, dict)}}

def check_requirements(weapon, strength, dexterity, intelligence=0, faith=0, arcane=0):
    r, err = _find_weapon(weapon)
    if err:
        return err
    have = {'Str': strength, 'Dex': dexterity, 'Int': intelligence, 'Fai': faith, 'Arc': arcane}
    need = weapon_req(r)
    missing = {k: need[k] - have.get(k, 0) for k in need if have.get(k, 0) < need[k]}
    return {'found': True, 'name': r['name'], 'can_wield': not missing, 'required': need, 'missing': missing}

calculate = calculate_trip_budget            # the Week 4 guarded runner, under a name that fits this domain

STAT_PROPS = {k: {'type': 'number'} for k in ('strength', 'dexterity', 'intelligence', 'faith', 'arcane')}
TOOLS = [
    {'name': 'get_weapon_stats',
     'description': 'Look up one weapon by its exact name. Returns weight, category, required stats and scaling as numbers.',
     'parameters': {'type': 'object', 'properties': {'weapon': {'type': 'string'}},
                    'required': ['weapon'], 'additionalProperties': False}},
    {'name': 'check_requirements',
     'description': ('Check whether a player with the given stats meets a weapon\'s stat requirements. Pass only the stats '
                     'the user stated; stats you leave out count as 0.'),
     'parameters': {'type': 'object', 'properties': {'weapon': {'type': 'string'}, **STAT_PROPS},
                    'required': ['weapon', 'strength', 'dexterity'], 'additionalProperties': False}},
    {'name': 'calculate',
     'description': ('Evaluate one arithmetic expression, for example the sum or difference of two weapon weights. '
                     'Guarded: numbers and + - * / ** and parentheses only, no names, calls or imports. '
                     'Use plain numbers you already got from other tool results.'),
     'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string'}},
                    'required': ['expression'], 'additionalProperties': False}},
]
IMPL = {'get_weapon_stats': get_weapon_stats, 'check_requirements': check_requirements, 'calculate': calculate}

# ---- copied verbatim from the Week 4 notebook (validator + dispatcher) ----
# ---------- validator + dispatcher (same discipline as the starter) ----------
class ToolArgError(Exception):
    pass

def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name'] == name), None)
    if spec is None:
        raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict):
        raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required', []):
        if r not in args:
            raise ToolArgError(f'missing required field: {r}')
    for k, v in args.items():
        p = spec['properties'].get(k)
        if p is None:
            raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str):
                raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float):
                raise ToolArgError(f'{k} must be a number (not a boolean)')
            if isinstance(v, float) and not math.isfinite(v):
                raise ToolArgError(f'{k} must be finite')
        else:
            raise ValueError(f'extend validate() to support schema type: {p["type"]}')
        if 'enum' in p and v not in p['enum']:
            raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')

def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)
        return {'ok': True, 'tool': name, 'output': output}
    except ToolArgError as e:
        return {'ok': False, 'tool': name, 'error_type': 'invalid_arguments', 'message': str(e)}
    except GuardrailViolation as e:
        return {'ok': False, 'tool': name, 'error_type': 'guardrail_blocked', 'message': str(e)}
    except Exception as e:
        return {'ok': False, 'tool': name, 'error_type': 'execution_error', 'message': str(e)}

def to_anthropic_tools(tools):
    # Anthropic's Messages API wants 'input_schema', not 'parameters' (the Week 4 fix).
    return [{'name': t['name'], 'description': t['description'], 'input_schema': t['parameters']} for t in tools]
ANTHROPIC_TOOLS = to_anthropic_tools(TOOLS)

_any = next(iter(WEAPON_BY_NAME.values()))[0]['name']
print('happy path:', dispatch('get_weapon_stats', {'weapon': _any}))
print('missing weapon:', dispatch('get_weapon_stats', {'weapon': 'Moonlight Spatula'}))
print('bad args:      ', dispatch('check_requirements', {'weapon': _any, 'strength': 'lots', 'dexterity': 10}))
print('guarded:       ', dispatch('calculate', {'expression': "__import__('os').system('echo hi')"}))
print('guarded:       ', dispatch('calculate', {'expression': '9**9**9**9**9'}))

happy path: {'ok': True, 'tool': 'get_weapon_stats', 'output': {'found': True, 'name': 'Hand Axe', 'category': 'Axe', 'weight': 3.5, 'required': {'Str': 9, 'Dex': 8}, 'scaling': {'Str': 'D', 'Dex': 'D'}}}
missing weapon: {'ok': True, 'tool': 'get_weapon_stats', 'output': {'found': False, 'weapon': 'Moonlight Spatula', 'suggestions': []}}
bad args:       {'ok': False, 'tool': 'check_requirements', 'error_type': 'invalid_arguments', 'message': 'strength must be a number (not a boolean)'}
guarded:        {'ok': False, 'tool': 'calculate', 'error_type': 'guardrail_blocked', 'message': 'disallowed expression element: Call'}
guarded:        {'ok': False, 'tool': 'calculate', 'error_type': 'guardrail_blocked', 'message': 'exponent too large'}


### 1c. The router (Week 7: retrieval-selected dynamic few-shot)

Every question goes down one of two paths. **lookup**: the answer is a fact written in one record (a
stat, a location, a drop). **tool**: the answer needs a check or a calculation (comparing or adding
stats, or testing a weapon against the player's own stats). The router is the Week 7 method: embed the
question, retrieve its 4 most similar *labeled* questions, put them in the prompt, ask Haiku for the label.

The labeled questions are generated from templates over real weapon and boss names. Entities are split
three ways by a hash of the name (about 50% router-train, 25% router-eval, 25% end-to-end eval), and the
router-eval and end-to-end questions use **phrasings the router never saw in training**. That is the
same guard against flattering few-shot as in Week 7.

In [ ]:
# ---- entity pools, split by a hash of the name so train / router-eval / end-to-end never share an entity ----
def part_of(name):
    h = int(hashlib.md5(f'{SEED}:{norm_name(name)}'.encode()).hexdigest(), 16) % 4
    return 'train' if h < 2 else ('router_eval' if h == 2 else 'e2e')

RUNE = re.compile(r'\brunes?\b', re.I)
def real_drops(rec):
    return [str(d).strip() for d in (rec.get('drops') or []) if str(d).strip() and not RUNE.search(str(d))]

USABLE_W, USABLE_WSD, USABLE_B = [], [], []
for c in CHUNKS:
    if NAME_COUNT[(c['kind'], norm_name(c['name']))] != 1:
        continue                                           # ambiguous names are not used for questions
    if c['kind'] == 'weapon':
        req = weapon_req(c['rec'])
        if num(c['rec'].get('weight')) is None or not req:
            continue
        USABLE_W.append(c)
        if set(req) <= {'Str', 'Dex'}:
            USABLE_WSD.append(c)
    elif c['kind'] == 'boss':
        loc = str(c['rec'].get('location') or c['rec'].get('region') or '').strip()
        if loc and 1 <= len(real_drops(c['rec'])) <= 3:
            USABLE_B.append(c)

def pool(items, part):
    return sorted((c for c in items if part_of(c['name']) == part), key=lambda c: c['name'])
POOLS = {p: {'w': pool(USABLE_W, p), 'wsd': pool(USABLE_WSD, p), 'b': pool(USABLE_B, p)}
         for p in ('train', 'router_eval', 'e2e')}
print({p: {k: len(v) for k, v in d.items()} for p, d in POOLS.items()})
need = {'train': (10, 10, 10), 'router_eval': (6, 6, 6), 'e2e': (12, 8, 8)}
for p, (w, wsd, b) in need.items():
    assert len(POOLS[p]['w']) >= w and len(POOLS[p]['wsd']) >= wsd and len(POOLS[p]['b']) >= b, \
        f'not enough usable entities in the {p} pool ({ {k: len(v) for k, v in POOLS[p].items()} }); the API data may be too small or have changed shape'

{'train': {'w': 154, 'wsd': 102, 'b': 34}, 'router_eval': {'w': 74, 'wsd': 50, 'b': 27}, 'e2e': {'w': 76, 'wsd': 44, 'b': 24}}


In [ ]:
# ---- labeled routing questions from templates; train and eval use DIFFERENT phrasings ----
LOOKUP_TRAIN = ['What are the stat requirements for the {w}?', 'How much does the {w} weigh?', 'Where do I fight {b}?',
                'What does {b} drop?', 'What category of weapon is the {w}?', 'Tell me about {b}.',
                'Describe the {w}.', 'Which attributes does the {w} scale with?', 'In which region can I find {b}?',
                'What is the weight of the {w}?']
TOOL_TRAIN = ['I have {s} Str and {d} Dex. Can I wield the {w}?', 'How much do the {w} and the {w2} weigh together?',
              'How much heavier is the {w} than the {w2}?', 'Do {s} Strength and {d} Dexterity meet the requirements for the {w}?',
              'What is the combined weight of the {w} and the {w2}?', "What's the weight difference between the {w} and the {w2}?",
              'If I carry the {w} and the {w2}, what is my total weight?', 'Am I strong and dexterous enough for the {w} with {s} Str and {d} Dex?']
LOOKUP_EVAL = ['What Strength and Dexterity does the {w} demand?', 'How heavy is the {w}?', 'Where is {b} located?',
               'What loot do I get from beating {b}?', 'What kind of weapon is the {w}?', 'Give me a quick description of {b}.']
TOOL_EVAL = ['With {s} Str and {d} Dex, am I able to equip the {w}?', 'Would the {w} be usable at {s} Strength and {d} Dexterity?',
             'Add up the weights of the {w} and the {w2}.', 'What is the total load of the {w} plus the {w2}?',
             'By how much does the {w} outweigh the {w2}?', 'Could someone with {s} Str and {d} Dex swing the {w}?']
assert not (set(LOOKUP_TRAIN + TOOL_TRAIN) & set(LOOKUP_EVAL + TOOL_EVAL))

def make_questions(templates, label, part, per_template, rng):
    w_pool = POOLS[part]['w'][:]; b_pool = POOLS[part]['b'][:]
    rng.shuffle(w_pool); rng.shuffle(b_pool)
    wi = bi = 0
    out, seen = [], set()
    for t in templates:
        made = 0
        while made < per_template:
            slots, ents = {}, []
            if '{w}' in t:
                c = w_pool[wi % len(w_pool)]; wi += 1; slots['w'] = c['name']; ents.append(c['id'])
            if '{w2}' in t:
                c = w_pool[wi % len(w_pool)]; wi += 1; slots['w2'] = c['name']; ents.append(c['id'])
            if '{b}' in t:
                c = b_pool[bi % len(b_pool)]; bi += 1; slots['b'] = c['name']; ents.append(c['id'])
            if '{s}' in t:
                slots['s'] = rng.randint(8, 30); slots['d'] = rng.randint(8, 30)
            q = t.format(**slots)
            if q in seen:
                continue
            seen.add(q); made += 1
            out.append({'q': q, 'label': label, 'ents': ents})
    return out

rng_r = random.Random(SEED)
ROUTER_TRAIN = make_questions(LOOKUP_TRAIN, 'lookup', 'train', 4, rng_r) + make_questions(TOOL_TRAIN, 'tool', 'train', 5, rng_r)
ROUTER_EVAL = make_questions(LOOKUP_EVAL, 'lookup', 'router_eval', 5, rng_r) + make_questions(TOOL_EVAL, 'tool', 'router_eval', 5, rng_r)
print('router train:', len(ROUTER_TRAIN), dict(Counter(r['label'] for r in ROUTER_TRAIN)),
      '| router eval:', len(ROUTER_EVAL), dict(Counter(r['label'] for r in ROUTER_EVAL)))
for r in ROUTER_TRAIN[:2] + ROUTER_TRAIN[-2:] + ROUTER_EVAL[:2] + ROUTER_EVAL[-2:]:
    print(f"  [{r['label']:6s}] {r['q']}")

router train: 80 {'lookup': 40, 'tool': 40} | router eval: 60 {'lookup': 30, 'tool': 30}
  [lookup] What are the stat requirements for the Thorned Whip?
  [lookup] What are the stat requirements for the Bastard's Stars?
  [tool  ] Am I strong and dexterous enough for the Grossmesser with 19 Str and 24 Dex?
  [tool  ] Am I strong and dexterous enough for the Pike with 13 Str and 17 Dex?
  [lookup] What Strength and Dexterity does the Sword Of Milos demand?
  [lookup] What Strength and Dexterity does the Lusat's Glintstone Staff demand?
  [tool  ] Could someone with 14 Str and 18 Dex swing the Lordsworn's Straight Sword?
  [tool  ] Could someone with 30 Str and 12 Dex swing the Eclipse Shotel?


In [ ]:
# ---- the router: Week 7's dynamic few-shot, plus the baselines it is compared with ----
ROUTES = ['lookup', 'tool']
R_TRAIN_VEC = embedder.encode([r['q'] for r in ROUTER_TRAIN], normalize_embeddings=True).astype('float32')
R_K = 4

def neighbors(q, k=R_K):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')[0]
    sims = R_TRAIN_VEC @ qe
    order = np.argsort(-sims)[:k]
    return [int(i) for i in order]

def router_prompt(q, shots=None):
    head = ('You route questions about Elden Ring game data to one of two paths.\n'
            'lookup = the answer is a fact written in one record (a stat, a location, a drop, a description).\n'
            'tool = the answer needs a check or a calculation: comparing or adding stats from several records, '
            "or testing a weapon against the player's own stats.\n"
            'Reply with only one word: lookup or tool.\n\n')
    body = ''.join(f'Question: {s["q"]}\nPath: {s["label"]}\n\n' for s in (shots or []))
    return head + body + f'Question: {q}\nPath:'

def parse_route(raw):
    if raw is None:
        return 'invalid'
    s = raw.strip().lower()
    for r in ROUTES:
        if re.match(rf'^{r}\b', s):
            return r
    hits = sorted((m.start(), r) for r in ROUTES for m in re.finditer(rf'\b{r}\b', s))
    return hits[0][1] if hits else 'invalid'

CALLS = {}                  # every (kind, key) is called exactly once; failures are not cached
def route_llm(variant, q, shots=None):
    key = ('route', variant, q)
    if key not in CALLS or CALLS[key]['error']:
        r, err = create(GEN_MODEL, max_tokens=8, messages=[{'role': 'user', 'content': router_prompt(q, shots)}])
        raw = r.content[0].text if r is not None else None
        CALLS[key] = {'raw': raw, 'pred': parse_route(raw), 'error': err,
                      'in_tok': r.usage.input_tokens if r is not None else 0}
    return CALLS[key]

def route_few_shot(q):
    return route_llm('few', q, [ROUTER_TRAIN[i] for i in neighbors(q)])

def route_knn(q):                 # plurality vote of the same 4 neighbors, ties to the nearest; no model call
    nb = neighbors(q); votes = Counter(ROUTER_TRAIN[i]['label'] for i in nb)
    top = max(votes.values()); tied = [l for l, v in votes.items() if v == top]
    return next(ROUTER_TRAIN[i]['label'] for i in nb if ROUTER_TRAIN[i]['label'] in tied)

def route_keyword_starter(q):     # the starter's router, verbatim
    return 'tool' if any(w in q.lower() for w in ['how many', 'fit', 'calculate', 'per ']) else 'lookup'

# Written AFTER seeing the eval phrasings, so it is optimistic: it shows what hand-tuned keywords can do here.
DOMAIN_WORDS = ['can i', 'able to', 'usable', 'together', 'heavier', 'combined', 'total', 'difference', 'add up',
                'outweigh', 'plus', 'meet the requirements', 'enough for', 'could someone', 'swing']
def route_keyword_domain(q):
    return 'tool' if any(w in q.lower() for w in DOMAIN_WORDS) else 'lookup'
print('router ready. neighbors of one eval question:')
_q = ROUTER_EVAL[0]['q']; print(' ', _q)
for i in neighbors(_q):
    print(f"    [{ROUTER_TRAIN[i]['label']:6s}] {ROUTER_TRAIN[i]['q']}")

router ready. neighbors of one eval question:
  What Strength and Dexterity does the Sword Of Milos demand?
    [tool  ] Do 17 Strength and 21 Dexterity meet the requirements for the Vyke's War Spear?
    [tool  ] Do 21 Strength and 26 Dexterity meet the requirements for the Mantis Blade?
    [lookup] Which attributes does the Rotten Crystal Sword scale with?
    [tool  ] How much heavier is the Watchdog's Staff than the Cane Sword?


### 1d. The grounded prompt (Week 3) and the assembled assistant

The lookup path is the Week 5 generation step: put the retrieved records in the prompt and tell the
model to answer from them only. The tool path is the Week 4 loop with the same grounding rules and the
same records as context. Both end in one short answer that names the entity.

In [ ]:
GROUNDED_RULES = ('Answer only from the context (and from tool results, if you used tools). Do not use outside knowledge '
                  'about Elden Ring. If the context does not contain what was asked, say "I don\'t have that in the data." '
                  'and, if a close match is in the context, name it. Keep it to one or two sentences and name the entity you are talking about.')
TOOL_RULES = (' For a weapon\'s stats use get_weapon_stats, to test a weapon against the player\'s stats use check_requirements, '
              'and for any arithmetic use calculate -- do not do arithmetic in prose. Pass check_requirements only the stats the user stated. '
              'If a tool says a weapon was not found, say so instead of guessing.')

def context_block(ids):
    return '\n'.join(f"- {CHUNKS[i]['text']}" for i in ids)

def generate_lookup(q, ids):
    """Week 3/5 grounded generation: one call, context in the prompt."""
    key = ('gen', q, tuple(ids))
    if key not in CALLS or CALLS[key]['error']:
        r, err = create(GEN_MODEL, max_tokens=300, system=GROUNDED_RULES,
                        messages=[{'role': 'user', 'content': f'Context:\n{context_block(ids)}\n\nQuestion: {q}\nAnswer:'}])
        CALLS[key] = {'text': r.content[0].text if r is not None else None, 'error': err}
    return CALLS[key]['text'], CALLS[key]['error']

def run_with_tools(q, ids, max_turns=6):
    """The Week 4 loop: send the query with tools attached, run whatever tools Claude asks for through
    dispatch(), feed the results back, repeat. Returns (text, call_log, error)."""
    key = ('tools', q, tuple(ids))
    if key in CALLS and not CALLS[key]['error']:
        c = CALLS[key]; return c['text'], c['log'], None
    messages = [{'role': 'user', 'content': f'Context:\n{context_block(ids)}\n\nQuestion: {q}'}]
    log, text, error = [], None, None
    for _ in range(max_turns):
        resp, err = create(GEN_MODEL, max_tokens=1024, system=GROUNDED_RULES + TOOL_RULES,
                           tools=ANTHROPIC_TOOLS, messages=messages)
        if resp is None:
            error = err; break
        messages.append({'role': 'assistant', 'content': resp.content})
        if resp.stop_reason != 'tool_use':
            text = ''.join(b.text for b in resp.content if getattr(b, 'type', None) == 'text'); break
        results = []
        for block in resp.content:
            if getattr(block, 'type', None) != 'tool_use':
                continue
            out = dispatch(block.name, block.input)
            log.append({'tool': block.name, 'args': dict(block.input), 'result': out})
            results.append({'type': 'tool_result', 'tool_use_id': block.id, 'content': json.dumps(out)})
        messages.append({'role': 'user', 'content': results})
    else:
        error = f'stopped after {max_turns} turns without a final answer -- possible loop'
    CALLS[key] = {'text': text, 'log': log, 'error': error}
    return text, log, error

def assistant(q, route=None, ids=None, retriever=None):
    """question -> route -> retrieve -> (grounded generation | tool loop) -> answer record."""
    r = route or route_few_shot(q)['pred']
    used = r if r in ROUTES else 'lookup'              # an unparseable route falls back to lookup, and is counted as wrong when scored
    ids = list(ids) if ids is not None else (retriever or retrieve)(q, K)
    if used == 'tool':
        text, log, err = run_with_tools(q, ids)
    else:
        (text, err), log = generate_lookup(q, ids), []
    return {'q': q, 'route': r, 'used': used, 'ids': ids, 'answer': text, 'log': log, 'error': err}

if API_OK:
    for q in ['Where is the Hand Axe, and what does it need?', 'How much do the Hand Axe and the Jawbone Axe weigh together?']:
        out = assistant(q)
        print(f"Q: {q}\n  route={out['route']}  retrieved={[CHUNKS[i]['name'] for i in out['ids']]}\n  tools={[l['tool'] for l in out['log']]}\n  A: {out['answer']}\n")
else:
    print('skipped (no API access)')

Q: Where is the Hand Axe, and what does it need?
  route=tool  retrieved=['Hand Axe', 'Jawbone Axe', 'Highland Axe', 'Battle Axe', 'Halberd']
  tools=[]
  A: Based on the context provided, the **Hand Axe** is a weapon in the game with the following requirements:

**Requirements:** Strength 9, Dexterity 8

**Additional details:** It's an Axe category weapon weighing 3.5, with Physical Attack 113 and Critical 100. It scales with Strength D and Dexterity D. The description notes it's commonly known as a hatchet and excels in successive attacks despite its short reach.

However, I don't have information about where to find or obtain the Hand Axe in the game world.

Q: How much do the Hand Axe and the Jawbone Axe weigh together?
  route=tool  retrieved=['Jawbone Axe', 'Battle Axe', 'Warped Axe', 'Hookclaws', 'Bastard Sword']
  tools=[]
  A: I don't have the Hand Axe in the data. The closest match is the Jawbone Axe, which weighs 5.5.



## Part 2: Evaluate

Four things are measured, in this order: **routing accuracy** (60 held-out questions, no judge needed),
**retrieval hit rate** (does the right record come back?), **end-to-end correctness on 18 held-out
cases** checked by plain code, and then the **LLM-as-judge** pass on those same 18 cases.

In [ ]:
def wilson(k, n, z=1.96):
    """95% Wilson score interval for a proportion k/n."""
    if n == 0:
        return 0.0, 0.0
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d

def sign_test_p(b, c):
    """Two-sided exact sign test on the disagreements (b = A only right, c = B only right)."""
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)

def fmt_ci(k, n):
    lo, hi = wilson(k, n)
    return f'{k / n:6.1%} [{lo:5.1%}, {hi:5.1%}]'

# self-checks of the helpers against known values (same as Week 7)
assert sign_test_p(0, 0) == 1.0 and abs(sign_test_p(5, 0) - 0.0625) < 1e-9 and sign_test_p(3, 3) == 1.0
assert abs(sign_test_p(7, 3) - 0.34375) < 1e-9
assert abs(wilson(50, 100)[0] - 0.4038) < 1e-3 and abs(wilson(50, 100)[1] - 0.5962) < 1e-3
print('stat helpers OK')

stat helpers OK


In [ ]:
# ---- routing accuracy on the held-out phrasings ----
gold_r = [r['label'] for r in ROUTER_EVAL]
ROUTE_PREDS = {
    'starter keyword router': [route_keyword_starter(r['q']) for r in ROUTER_EVAL],
    'domain keywords (written after seeing eval phrasings)': [route_keyword_domain(r['q']) for r in ROUTER_EVAL],
    f'kNN vote of {R_K} labeled questions (no LLM)': [route_knn(r['q']) for r in ROUTER_EVAL],
}
if API_OK:
    t0 = time.time(); failed = 0
    zero, few = [], []
    for n, r in enumerate(ROUTER_EVAL):
        if n % 20 == 0:
            print(f'  router eval {n}/{len(ROUTER_EVAL)}   {time.time() - t0:.0f}s elapsed   {failed} failed calls so far', flush=True)
        z = route_llm('zero', r['q']); f = route_few_shot(r['q'])
        failed += bool(z['error']) + bool(f['error'])
        zero.append(z); few.append(f)
        if failed >= 6 and n < 6:
            print('Too many failed calls; stopping the router eval. Check the key / model above.'); break
    if len(few) == len(ROUTER_EVAL) and not any(c['error'] for c in zero + few):
        ROUTE_PREDS['zero-shot Haiku (no examples)'] = [c['pred'] for c in zero]
        ROUTE_PREDS[f'dynamic {R_K}-shot Haiku (the router used below)'] = [c['pred'] for c in few]
    else:
        print('WARNING: some router calls failed, so the LLM rows are left out. Re-run this cell.')
print(f'{"router":58s} accuracy [95% CI]')
for name, preds in ROUTE_PREDS.items():
    k = sum(p == g for p, g in zip(preds, gold_r))
    print(f'{name:58s} {fmt_ci(k, len(gold_r))}')
k_key = f'dynamic {R_K}-shot Haiku (the router used below)'
if k_key in ROUTE_PREDS:
    fs, zs = ROUTE_PREDS[k_key], ROUTE_PREDS['zero-shot Haiku (no examples)']
    b = sum(f == g and z != g for f, z, g in zip(fs, zs, gold_r)); c = sum(f != g and z == g for f, z, g in zip(fs, zs, gold_r))
    print(f'\nfew-shot vs zero-shot: few-shot only right {b}, zero-shot only right {c}; exact sign test p = {sign_test_p(b, c):.3f}')
    print('invalid replies (few-shot):', sum(1 for p in fs if p == 'invalid'))
    wrong = [(r['q'], r['label'], p) for r, p in zip(ROUTER_EVAL, fs) if p != r['label']]
    print('router mistakes:', wrong[:8] if wrong else 'none')

  router eval 0/60   0s elapsed   0 failed calls so far
  router eval 20/60   19s elapsed   0 failed calls so far
  router eval 40/60   39s elapsed   0 failed calls so far
router                                                     accuracy [95% CI]
starter keyword router                                      50.0% [37.7%, 62.3%]
domain keywords (written after seeing eval phrasings)      100.0% [94.0%, 100.0%]
kNN vote of 4 labeled questions (no LLM)                    85.0% [73.9%, 91.9%]
zero-shot Haiku (no examples)                              100.0% [94.0%, 100.0%]
dynamic 4-shot Haiku (the router used below)               100.0% [94.0%, 100.0%]

few-shot vs zero-shot: few-shot only right 0, zero-shot only right 0; exact sign test p = 1.000
invalid replies (few-shot): 0
router mistakes: none


### End-to-end cases (held out) and plain-code scoring

18 questions, built from the **end-to-end entity pool and the held-out phrasings**, with the gold answer
computed from the records before any model call: stat requirements (3), a weight lookup (1), boss
location (3), boss drops (2), "can I wield it?" with the player's stats (3), weight sums and a difference
(3), and **3 questions the data cannot answer** (a weapon that does not exist, once on each path, and a
drop rate the data does not contain). The unanswerable ones test whether the assistant declines instead of
inventing.

Scoring by plain code, so the judge has something independent to be compared with:
numbers and attribute names present (requirements), the stated location or region present, every
non-rune drop named, the correct number present (weights), the **tool trace** for "can I wield it"
(the right weapon was checked and the tool's verdict matches the gold answer), and a decline phrase for
the unanswerable ones. These checks are crude; they are a floor, not a verdict.

In [ ]:
def fmtn(x):
    x = float(x); return str(int(x)) if x == int(x) else str(round(x, 2))

def squash(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

rng_e = random.Random(SEED + 100)
E = POOLS['e2e']
wsd = E['wsd'][:]; wall = E['w'][:]; bos = E['b'][:]
rng_e.shuffle(wsd); rng_e.shuffle(wall); rng_e.shuffle(bos)

CASES = []
def add(type_, q, gold_route, gold_ids, gold_text, check):
    CASES.append({'id': len(CASES), 'type': type_, 'q': q, 'gold_route': gold_route, 'gold_ids': gold_ids,
                  'gold_text': gold_text, 'check': check})

for c in wsd[:3]:                                                  # stat requirements
    req = weapon_req(c['rec'])
    add('requirements', LOOKUP_EVAL[0].format(w=c['name']), 'lookup', [c['id']],
        f"{c['name']} requires " + ', '.join(f'{k} {fmtn(v)}' for k, v in req.items()) + '.', {'attrs': req})
c = wall[0]                                                         # weight lookup
add('weight_lookup', LOOKUP_EVAL[1].format(w=c['name']), 'lookup', [c['id']],
    f"{c['name']} weighs {fmtn(num(c['rec']['weight']))}.", {'number': float(num(c['rec']['weight']))})
for c in bos[:3]:                                                   # boss location
    loc = str(c['rec'].get('location') or '').strip(); reg = str(c['rec'].get('region') or '').strip()
    add('boss_location', LOOKUP_EVAL[2].format(b=c['name']), 'lookup', [c['id']],
        f"{c['name']} is at {loc or reg}" + (f' ({reg})' if loc and reg else '') + '.', {'places': [p for p in (loc, reg) if p]})
for c in bos[3:5]:                                                  # boss drops
    d = real_drops(c['rec'])
    add('boss_drops', LOOKUP_EVAL[3].format(b=c['name']), 'lookup', [c['id']],
        f"{c['name']} drops " + '; '.join(d) + '.', {'drops': d})
tmpl_cw = [TOOL_EVAL[0], TOOL_EVAL[1], TOOL_EVAL[5]]                 # can I wield it?
cw_pool = wsd[3:]
for j, want in enumerate([True, False, True]):
    # a "cannot wield" case needs a requirement of at least 4 so it can be lowered and still be a clean fail
    c = next(x for x in cw_pool if want or max(weapon_req(x['rec']).values()) >= 4)
    cw_pool.remove(c)
    req = weapon_req(c['rec'])
    st = {'Str': 0, 'Dex': 0}
    if want:
        for k in st: st[k] = req.get(k, 0) + rng_e.randint(0, 3)
        st = {k: max(v, 5) for k, v in st.items()}
    else:
        low = max(req, key=lambda k: req[k])
        for k in st: st[k] = req.get(k, 0) + rng_e.randint(0, 3)
        st[low] = max(1, req[low] - rng_e.randint(1, 3))
    can = all(st.get(k, 0) >= v for k, v in req.items())
    assert can == want, (c['name'], req, st)
    add('can_wield', tmpl_cw[j].format(s=st['Str'], d=st['Dex'], w=c['name']), 'tool', [c['id']],
        f"With {st['Str']} Str and {st['Dex']} Dex: {c['name']} (needs {', '.join(f'{k} {fmtn(v)}' for k, v in req.items())}) "
        f"{'can' if can else 'cannot'} be wielded.", {'weapon': c['name'], 'can': can})
pairs_ = [(wall[1], wall[2]), (wall[3], wall[4]), (wall[5], wall[6])]
for (c1, c2), tm, kind in zip(pairs_, [TOOL_EVAL[2], TOOL_EVAL[3], TOOL_EVAL[4]], ['sum', 'sum', 'diff']):
    w1, w2 = float(num(c1['rec']['weight'])), float(num(c2['rec']['weight']))
    if kind == 'diff' and w1 < w2:
        c1, c2, w1, w2 = c2, c1, w2, w1                              # the weapon named first is the heavier one
    val = w1 + w2 if kind == 'sum' else w1 - w2
    add('weight_math', tm.format(w=c1['name'], w2=c2['name']), 'tool', [c1['id'], c2['id']],
        f"{c1['name']} ({fmtn(w1)}) and {c2['name']} ({fmtn(w2)}): {'total' if kind == 'sum' else 'difference'} {fmtn(val)}.",
        {'number': round(val, 2)})
FAKE_W = ['Frostbitten Kitchen Knife', 'Hollow Oakwood Spear']
assert all(norm_name(n) not in WEAPON_BY_NAME for n in FAKE_W)
add('unanswerable', LOOKUP_TRAIN[0].format(w=FAKE_W[0]), 'lookup', [], f'{FAKE_W[0]} is not in the data; the assistant should say so.', {})
add('unanswerable', TOOL_EVAL[0].format(s=20, d=12, w=FAKE_W[1]), 'tool', [], f'{FAKE_W[1]} is not in the data; the assistant should say so.', {})
c = bos[5]; d0 = real_drops(c['rec'])[0]
add('unanswerable', f'What is the drop rate of {d0} from {c["name"]}?', 'lookup', [c['id']],
    f'The data lists {d0} as a drop but gives no drop rate; the assistant should say it does not have the rate.', {})

print(len(CASES), 'cases:', dict(Counter(c['type'] for c in CASES)))
for c in CASES:
    print(f"  {c['id']:2d} [{c['gold_route']:6s}] {c['type']:13s} {c['q']}")
    print(f"       gold: {c['gold_text']}")

18 cases: {'requirements': 3, 'weight_lookup': 1, 'boss_location': 3, 'boss_drops': 2, 'can_wield': 3, 'weight_math': 3, 'unanswerable': 3}
   0 [lookup] requirements  What Strength and Dexterity does the Pest's Glaive demand?
       gold: Pest's Glaive requires Str 13, Dex 13.
   1 [lookup] requirements  What Strength and Dexterity does the Cleanrot Spear demand?
       gold: Cleanrot Spear requires Str 16.
   2 [lookup] requirements  What Strength and Dexterity does the Large Club demand?
       gold: Large Club requires Str 22.
   3 [lookup] weight_lookup How heavy is the Winged Greathorn?
       gold: Winged Greathorn weighs 11.
   4 [lookup] boss_location Where is Demi-human Queen Margot located?
       gold: Demi-human Queen Margot is at Mt. Gelmir (Mount Gelmir).
   5 [lookup] boss_location Where is Ulcerated Tree Spirit located?
       gold: Ulcerated Tree Spirit is at Mt. Gelmir (Mount Gelmir).
   6 [lookup] boss_location Where is Draconic Tree Sentinel located?
       gold: D

In [ ]:
DECLINE = ["don't have", 'do not have', 'not in the data', 'no information', "couldn't find", 'could not find', 'not found',
           "isn't in", 'is not in', 'unable to', 'cannot find', "can't find", 'no record', 'not available',
           "doesn't contain", 'does not contain', 'not specified', "doesn't specify", 'does not specify',
           "doesn't list", 'does not list', "doesn't include", 'does not include', 'no such', 'not listed']
def numbers_in(text):
    return [float(x) for x in re.findall(r'(?<![\w.])\d+(?:\.\d+)?', text or '')]
def has_number(text, v, tol=0.011):
    return any(abs(n - float(v)) < tol for n in numbers_in(text))

def det_check(case, res):
    """Plain-code correctness (crude on purpose). Returns True / False."""
    if res['answer'] is None and not res['log']:
        return False
    ans = res['answer'] or ''; a = ans.lower(); t = case['type']; ck = case['check']
    if t == 'requirements':
        return all(k.lower() in a and has_number(ans, v, 0.01) for k, v in ck['attrs'].items()) or \
               all(({'Str': 'strength', 'Dex': 'dexterity'}.get(k, k)).lower() in a and has_number(ans, v, 0.01) for k, v in ck['attrs'].items())
    if t in ('weight_lookup', 'weight_math'):
        return has_number(ans, ck['number'])
    if t == 'boss_location':
        return any(squash(p) in squash(ans) for p in ck['places'])
    if t == 'boss_drops':
        return all(squash(d) in squash(ans) for d in ck['drops'])
    if t == 'can_wield':                                   # judged on the TOOL TRACE: right weapon checked, verdict matches gold
        return any(l['tool'] == 'check_requirements' and l['result'].get('ok') and squash(l['args'].get('weapon', '')) == squash(ck['weapon'])
                   and l['result']['output'].get('can_wield') == ck['can'] for l in res['log'])
    if t == 'unanswerable':
        return any(m in a for m in DECLINE)
    return False

RESULTS = {}              # case id -> assistant record
def run_cases():
    t0 = time.time()
    for n, case in enumerate(CASES):
        if case['id'] in RESULTS and not RESULTS[case['id']]['error']:
            continue
        print(f"  case {n + 1}/{len(CASES)}   {time.time() - t0:.0f}s elapsed", flush=True)
        RESULTS[case['id']] = assistant(case['q'])
if API_OK:
    run_cases()
    bad = [i for i, r in RESULTS.items() if r['error']]
    if bad:
        print(f'WARNING: {len(bad)} cases hit API errors and are NOT scored; re-run this cell. First error: {RESULTS[bad[0]]["error"]}')
    SCORED = [c for c in CASES if c['id'] in RESULTS and not RESULTS[c['id']]['error']]
    for c in SCORED:
        c['det_ok'] = det_check(c, RESULTS[c['id']])
    print(f'\nend-to-end (plain-code checks): {fmt_ci(sum(c["det_ok"] for c in SCORED), len(SCORED))}  n = {len(SCORED)}')
    print(f'{"type":14s} right/total')
    for t in dict.fromkeys(c['type'] for c in SCORED):
        g = [c for c in SCORED if c['type'] == t]; print(f'{t:14s} {sum(c["det_ok"] for c in g)}/{len(g)}')
    rt = [c for c in SCORED if RESULTS[c['id']]['used'] == c['gold_route']]
    print(f'\nrouted correctly: {len(rt)}/{len(SCORED)}   retrieval had every gold record: '
          f'{sum(set(c["gold_ids"]) <= set(RESULTS[c["id"]]["ids"]) for c in SCORED)}/{len(SCORED)}')
else:
    SCORED = []; print('skipped (no API access)')

  case 1/18   0s elapsed
  case 2/18   1s elapsed
  case 3/18   3s elapsed
  case 4/18   4s elapsed
  case 5/18   5s elapsed
  case 6/18   7s elapsed
  case 7/18   8s elapsed
  case 8/18   9s elapsed
  case 9/18   11s elapsed
  case 10/18   13s elapsed
  case 11/18   15s elapsed
  case 12/18   18s elapsed
  case 13/18   20s elapsed
  case 14/18   22s elapsed
  case 15/18   24s elapsed
  case 16/18   26s elapsed
  case 17/18   28s elapsed
  case 18/18   30s elapsed

end-to-end (plain-code checks):  83.3% [60.8%, 94.2%]  n = 18
type           right/total
requirements   3/3
weight_lookup  1/1
boss_location  3/3
boss_drops     1/2
can_wield      2/3
weight_math    2/3
unanswerable   3/3

routed correctly: 18/18   retrieval had every gold record: 14/18


### The LLM-as-judge pass

The judge is a different Claude model (`JUDGE_MODEL`). It sees the question, a **reference answer built
from the records**, the evidence the assistant was given (retrieved records and any tool results), and the
assistant's answer, and scores three named dimensions from 1 to 5 through a forced tool call so the
output is always parseable:

- **correctness**: does the answer agree with the reference?
- **grounding**: is every factual claim in the answer supported by the evidence shown? Outside knowledge, even if true, lowers it.
- **calibration**: does it decline when the evidence does not cover the question, and answer when it does?

Limits to keep in mind: the judge is the same model *family*, 18 cases is small, and LLM judges tend to
prefer fluent, confident, longer answers. The comparison with the plain-code checks below is there to
show where the two disagree.

In [ ]:
JUDGE_SYSTEM = ('You are a strict grader of answers from an Elden Ring data assistant. Score each dimension from 1 (very poor) '
                'to 5 (excellent) by calling submit_scores.\n'
                '- correctness: does the answer agree with the REFERENCE? 5 = fully, 3 = partly or one detail wrong, 1 = wrong or missing. '
                'For a question the data cannot answer, the reference says so: a correct decline is 5.\n'
                '- grounding: is every factual claim supported by the EVIDENCE shown? Any claim that is not in the evidence '
                '(including true outside knowledge) lowers this. 5 = every claim supported.\n'
                '- calibration: does it decline or hedge when the evidence does not cover the question, and answer plainly when it does? '
                '5 = right call both ways; 1 = invents an answer it cannot support, or refuses when the evidence was there.')
JUDGE_TOOL = {'name': 'submit_scores', 'description': 'Submit the three scores (integers 1 to 5) and a one-sentence reason.',
              'input_schema': {'type': 'object', 'properties': {
                  'correctness': {'type': 'integer'}, 'grounding': {'type': 'integer'},
                  'calibration': {'type': 'integer'}, 'reason': {'type': 'string'}},
                  'required': ['correctness', 'grounding', 'calibration', 'reason']}}

def evidence_text(res, limit=3500):
    ctx = '\n'.join(f"- {CHUNKS[i]['text']}" for i in res['ids'])
    tools = '\n'.join(f"- {l['tool']}({json.dumps(l['args'])}) -> {json.dumps(l['result'].get('output', l['result']))}" for l in res['log'])
    return (f'RETRIEVED RECORDS:\n{ctx}\n' + (f'TOOL CALLS:\n{tools}\n' if tools else ''))[:limit]

JUDGED = {}
def judge(case, res):
    key = ('judge', case['id'], res['answer'], tuple(res['ids']))
    if key in CALLS and not CALLS[key]['error']:
        return CALLS[key]
    prompt = (f"QUESTION: {case['q']}\n\nREFERENCE ANSWER: {case['gold_text']}\n\nEVIDENCE THE ASSISTANT HAD:\n{evidence_text(res)}\n"
              f"\nASSISTANT'S ANSWER: {res['answer']}")
    r, err = create(JUDGE_MODEL, max_tokens=300, system=JUDGE_SYSTEM, tools=[JUDGE_TOOL],
                    tool_choice={'type': 'tool', 'name': 'submit_scores'}, messages=[{'role': 'user', 'content': prompt}])
    out = {'error': err, 'scores': None, 'reason': None}
    if r is not None:
        blk = next((b for b in r.content if getattr(b, 'type', None) == 'tool_use'), None)
        s = dict(blk.input) if blk is not None else {}
        if all(isinstance(s.get(d), int) and 1 <= s[d] <= 5 for d in ('correctness', 'grounding', 'calibration')):
            out['scores'] = {d: s[d] for d in ('correctness', 'grounding', 'calibration')}; out['reason'] = s.get('reason')
        else:
            out['error'] = f'judge returned unusable scores: {s}'
    CALLS[key] = out
    return out

if API_OK and JUDGE_OK and SCORED:
    for c in SCORED:
        j = judge(c, RESULTS[c['id']]); c['judge'] = j
    ok = [c for c in SCORED if c['judge']['scores']]
    if len(ok) < len(SCORED):
        print(f'WARNING: {len(SCORED) - len(ok)} judge calls failed/unusable; re-run this cell. e.g. {next(c["judge"]["error"] for c in SCORED if not c["judge"]["scores"])}')
    if not ok:
        print('No usable judge scores, so the judge report is skipped.')
    else:
        DIMS = ('correctness', 'grounding', 'calibration')
        print(f'judge: {JUDGE_MODEL}   cases judged: {len(ok)}/{len(SCORED)}\n')
        print(f'{"dimension":13s} mean   distribution (count of 1,2,3,4,5)')
        for d in DIMS:
            v = [c['judge']['scores'][d] for c in ok]
            print(f'{d:13s} {sum(v) / len(v):.2f}   {[v.count(x) for x in (1, 2, 3, 4, 5)]}')
        print(f'\n{"type":14s} ' + '  '.join(f'{d[:6]:>6s}' for d in DIMS) + '   (mean per question type)')
        for t in dict.fromkeys(c['type'] for c in ok):
            g = [c for c in ok if c['type'] == t]
            print(f'{t:14s} ' + '  '.join(f'{sum(c["judge"]["scores"][d] for c in g) / len(g):6.2f}' for d in DIMS))
        # does the judge agree with the plain-code check?
        both = sum(c['det_ok'] and c['judge']['scores']['correctness'] >= 4 for c in ok)
        det_only = sum(c['det_ok'] and c['judge']['scores']['correctness'] < 4 for c in ok)
        judge_only = sum((not c['det_ok']) and c['judge']['scores']['correctness'] >= 4 for c in ok)
        neither = len(ok) - both - det_only - judge_only
        print(f'\njudge (correctness >= 4) vs plain-code check: both pass {both}, only code passes {det_only}, only judge passes {judge_only}, both fail {neither}')
        low = [c for c in ok if min(c['judge']['scores'].values()) <= 3]
        print(f'\ncases with any score <= 3: {len(low)}')
        for c in low:
            print(f"  [{c['type']}] {c['q']}\n     scores {c['judge']['scores']}  -- {c['judge']['reason']}")
else:
    print('skipped (needs the generator, the judge model and scored cases)')

/tmp/ipykernel_3387/2803839108.py:48: DeprecationWarning: The model 'claude-sonnet-4-5-20250929' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  return _client.messages.create(model=model, **args), None


judge: claude-sonnet-4-5-20250929   cases judged: 18/18

dimension     mean   distribution (count of 1,2,3,4,5)
correctness   4.33   [3, 0, 0, 0, 15]
grounding     4.89   [0, 0, 1, 0, 17]
calibration   4.33   [3, 0, 0, 0, 15]

type           correc  ground  calibr   (mean per question type)
requirements     5.00    5.00    5.00
weight_lookup    5.00    5.00    5.00
boss_location    5.00    5.00    5.00
boss_drops       3.00    4.00    3.00
can_wield        3.67    5.00    3.67
weight_math      3.67    5.00    3.67
unanswerable     5.00    5.00    5.00

judge (correctness >= 4) vs plain-code check: both pass 15, only code passes 0, only judge passes 0, both fail 3

cases with any score <= 3: 3
  [boss_drops] What loot do I get from beating Godskin Apostle And Godskin Noble?
     scores {'correctness': 1, 'grounding': 3, 'calibration': 1}  -- The assistant answered about individual bosses when the question asked about "Godskin Apostle AND Godskin Noble" (a specific duo encounter), which 

## Part 3: One failure, traced to a component

The cells below do not assume which part is to blame. For every case that failed (plain-code check
failed, or any judge score of 3 or lower), they report the stage that broke first: **route**, **retrieval**,
**tool**, or **generation**. Then an ablation changes one component at a time and re-runs all 18 cases:

- **oracle route**: force the correct path instead of the router's choice
- **oracle retrieval**: hand the assistant exactly the gold records instead of what retrieval found
- **both**

A component is the cause when giving it the right answer fixes the case and nothing else does. Because
every call is cached, only the changed conditions cost new calls.

In [ ]:
def diagnose(case, res):
    """First broken stage, in pipeline order. A diagnosis from one run, confirmed (or not) by the ablation below."""
    if res['used'] != case['gold_route']:
        return 'route'
    if case['gold_ids'] and not set(case['gold_ids']) <= set(res['ids']):
        return 'retrieval'
    if res['used'] == 'tool' and any(not l['result'].get('ok') for l in res['log']):
        return 'tool'
    return 'generation'

def failed_cases():
    out = []
    for c in SCORED:
        low = c.get('judge', {}).get('scores') and min(c['judge']['scores'].values()) <= 3
        if not c['det_ok'] or low:
            out.append(c)
    return out

if SCORED:
    FAILS = failed_cases()
    print(f'{len(FAILS)} of {len(SCORED)} cases failed one of the checks')
    print('first broken stage:', dict(Counter(diagnose(c, RESULTS[c['id']]) for c in FAILS)))
    for c in FAILS:
        r = RESULTS[c['id']]
        print(f"\n[{c['id']}] {c['type']}: {c['q']}")
        print(f"   gold route {c['gold_route']}, routed {r['route']}; stage -> {diagnose(c, r)}")
        print(f"   gold records: {[CHUNKS[i]['name'] for i in c['gold_ids']]}")
        print(f"   retrieved:    {[CHUNKS[i]['name'] for i in r['ids']]}")
        if r['log']:
            print('   tools:        ' + ' | '.join(f"{l['tool']}({json.dumps(l['args'])}) -> {json.dumps(l['result'].get('output', l['result']))[:110]}" for l in r['log']))
        print(f"   reference: {c['gold_text']}\n   answer:    {r['answer']}")
        if c.get('judge', {}).get('scores'):
            print(f"   judge: {c['judge']['scores']} -- {c['judge']['reason']}")
else:
    print('skipped')

3 of 18 cases failed one of the checks
first broken stage: {'retrieval': 3}

[7] boss_drops: What loot do I get from beating Godskin Apostle And Godskin Noble?
   gold route lookup, routed lookup; stage -> retrieval
   gold records: ['Godskin Apostle And Godskin Noble']
   retrieved:    ['Godskin Noble', 'Godskin Apostle', 'Godskin Apostle', 'Godskin Duo', 'Godefroy The Grafted']
   reference: Godskin Apostle And Godskin Noble drops Godskin Swaddling Cloth x1; Black Flame Ritual x1.
   answer:    From **Godskin Apostle**, you get: 14,000 Runes, Godskin Peeler, Scouring Black Flame, 94,000 Runes, Godskin Apostle Hood, Godskin Apostle Robe, Godskin Apostle Bracers, Godskin Apostle Trousers, 70,000 Runes, Godskin Swaddling Cloth, and Black Flame Ritual.

From **Godskin Noble**, you get: 70,000 Runes, Godskin Swaddling Cloth, Black Flame Ritual, 50,000 Runes, Godskin Stitcher, and Noble Presence.
   judge: {'correctness': 1, 'grounding': 3, 'calibration': 1} -- The assistant answered about

In [ ]:
# ---- ablation: change ONE component at a time (all 18 cases per condition) ----
ABL = {}
def run_condition(name, route_oracle=False, retrieval_oracle=False):
    rows = {}
    for c in SCORED:
        res = assistant(c['q'], route=c['gold_route'] if route_oracle else None,
                        ids=(c['gold_ids'] or None) if retrieval_oracle else None)
        rows[c['id']] = (not res['error']) and det_check(c, res), res
    ABL[name] = rows

if SCORED:
    t0 = time.time()
    run_condition('full system')
    run_condition('oracle route', route_oracle=True)
    run_condition('oracle retrieval', retrieval_oracle=True)
    run_condition('oracle route + retrieval', route_oracle=True, retrieval_oracle=True)
    print(f'ablation done in {time.time() - t0:.0f}s\n')
    print(f'{"condition":28s} right (plain-code check) [95% CI]')
    for name, rows in ABL.items():
        k = sum(ok for ok, _ in rows.values()); print(f'{name:28s} {fmt_ci(k, len(rows))}')
    flip = [c for c in SCORED if not ABL['full system'][c['id']][0]]
    if flip:
        print(f'\n{"case":>4s}  {"type":13s}' + ''.join(f'{n[:18]:>20s}' for n in ABL))
        for c in flip:
            print(f'{c["id"]:>4d}  {c["type"]:13s}' + ''.join(f'{("pass" if ABL[n][c["id"]][0] else "FAIL"):>20s}' for n in ABL))
    else:
        print('\nno case failed under the plain-code check in the full system')
else:
    print('skipped')

ablation done in 21s

condition                    right (plain-code check) [95% CI]
full system                   83.3% [60.8%, 94.2%]
oracle route                  83.3% [60.8%, 94.2%]
oracle retrieval              94.4% [74.2%, 99.0%]
oracle route + retrieval      94.4% [74.2%, 99.0%]

case  type                  full system        oracle route    oracle retrieval  oracle route + ret
   7  boss_drops                   FAIL                FAIL                FAIL                FAIL
  11  can_wield                    FAIL                FAIL                pass                pass
  14  weight_math                  FAIL                FAIL                pass                pass


### Retrieval check: dense vs. dense + lexical re-rank (no model calls)

If the failures point at retrieval, the Week 6 re-ranker is the obvious candidate fix (over-retrieve 20,
re-score each as 0.5 x embedding similarity + 0.5 x word overlap, keep the top 5). Game names are
exactly the kind of rare token that word overlap catches and embeddings blur. This cell measures it on
all 60 router-eval questions plus the 18 cases, whether or not anything failed, so the comparison does not
depend on what went wrong. The question is whether **every record the question needs** is in the top 5.

In [ ]:
# ---- Week 6 re-ranker, reused (lexical_score / rerank unchanged except for indexing into CHUNKS) ----
STOP = set('a an the of to in on for and or is are how do does what so that with by as it why can does should would be will'.split())
def lexical_score(q, chunk):
    q_terms = set(re.findall(r'[a-zA-Z0-9]+', q.lower())) - STOP
    c_terms = set(re.findall(r'[a-zA-Z0-9]+', chunk.lower()))
    return len(q_terms & c_terms) / len(q_terms) if q_terms else 0.0

def retrieve_hybrid(q, k=K, over=20, alpha=0.5):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    cand = [int(i) for i in index.search(qe, over)[1][0] if i >= 0]
    scored = sorted(((alpha * float(cvec[i] @ qe[0]) + (1 - alpha) * lexical_score(q, CHUNKS[i]['text']), i) for i in cand),
                    key=lambda t: -t[0])
    return [i for _, i in scored[:k]]

PROBES = [(r['q'], r['ents']) for r in ROUTER_EVAL] + [(c['q'], c['gold_ids']) for c in CASES if c['gold_ids']]
dense_hit = [set(g) <= set(retrieve(q, K)) for q, g in PROBES]
hyb_hit = [set(g) <= set(retrieve_hybrid(q, K)) for q, g in PROBES]
b = sum(h and not d for d, h in zip(dense_hit, hyb_hit)); c_ = sum(d and not h for d, h in zip(dense_hit, hyb_hit))
print(f'questions with gold records: {len(PROBES)}')
print(f'dense (Week 5):        every gold record in top {K}: {fmt_ci(sum(dense_hit), len(PROBES))}')
print(f'dense + rerank (Wk 6): every gold record in top {K}: {fmt_ci(sum(hyb_hit), len(PROBES))}')
print(f'hybrid only right {b}, dense only right {c_}; exact sign test p = {sign_test_p(b, c_):.3f}')
miss = [q for (q, g), d in zip(PROBES, dense_hit) if not d]
print('\nsome questions where dense retrieval missed a gold record:')
for q in miss[:6]:
    print(' ', q)

questions with gold records: 76
dense (Week 5):        every gold record in top 5:  86.8% [77.4%, 92.7%]
dense + rerank (Wk 6): every gold record in top 5:  94.7% [87.2%, 97.9%]
hybrid only right 6, dense only right 0; exact sign test p = 0.031

some questions where dense retrieval missed a gold record:
  Add up the weights of the Pulley Bow and the Spiked Club.
  Add up the weights of the Broadsword and the Godskin Peeler.
  Add up the weights of the Ringed Finger and the Staff Of Loss.
  What is the total load of the Frozen Needle plus the Rapier?
  What is the total load of the Shotel plus the Shortbow?
  By how much does the Glintstone Kris outweigh the Lordsworn's Greatsword?


**Failure analysis (fill in after the run).**

**TODO (written after the Colab run):** name one failure from the list above, say which stage broke first
and what the ablation shows (did giving that one component the right answer fix it?), then say what the
mitigation is and whether the next cell measured it. Quote the actual question, the retrieved record
names, the tool trace and the answer. If the plain-code checks and the judge disagree about a case, say
which one you believe and why. If nothing failed at all, say that plainly and use the unanswerable cases
and the routing mistakes as the failure to analyze.

In [ ]:
# ---- mitigation test: re-run the failed cases with the Week 6 re-ranked retriever, same router ----
if SCORED and failed_cases():
    print(f'{"case":>4s}  {"type":13s}  before  after (dense + rerank)')
    fixed = 0
    for c in failed_cases():
        before = det_check(c, RESULTS[c['id']])
        res2 = assistant(c['q'], retriever=retrieve_hybrid)
        after = (not res2['error']) and det_check(c, res2); fixed += (not before) and after
        print(f'{c["id"]:>4d}  {c["type"]:13s}  {"pass" if before else "FAIL"}    {"pass" if after else "FAIL"}')
    print(f'\nfixed by re-ranking: {fixed} of {len(failed_cases())} failed cases (n is tiny; read it as an illustration, not a result)')
else:
    print('nothing to re-run')

case  type           before  after (dense + rerank)
   7  boss_drops     FAIL    pass
  11  can_wield      FAIL    FAIL
  14  weight_math    FAIL    pass

fixed by re-ranking: 2 of 3 failed cases (n is tiny; read it as an illustration, not a result)


In [ ]:
# ---- save a small results file for the README / research note ----
if SCORED:
    out = {'fingerprint': FINGERPRINT, 'chunks': len(CHUNKS), 'gen_model': GEN_MODEL, 'judge_model': JUDGE_MODEL,
           'temperature_pinned': TEMP_PINNED, 'k': K, 'n_cases': len(SCORED),
           'plain_code_ok': sum(c['det_ok'] for c in SCORED),
           'routing': {n: sum(p == g for p, g in zip(ps, gold_r)) for n, ps in ROUTE_PREDS.items()}, 'routing_n': len(gold_r),
           'ablation': {n: sum(ok for ok, _ in rows.values()) for n, rows in ABL.items()} if ABL else {},
           'cases': [{'id': c['id'], 'type': c['type'], 'q': c['q'], 'det_ok': c['det_ok'], 'route': RESULTS[c['id']]['route'],
                      'judge': (c.get('judge') or {}).get('scores')} for c in SCORED]}
    pathlib.Path('data').mkdir(exist_ok=True)
    pathlib.Path('data/week8_results.json').write_text(json.dumps(out, indent=2))
    print('saved data/week8_results.json (commit this; do NOT commit data/raw/)')

saved data/week8_results.json (commit this; do NOT commit data/raw/)


## Results

**TODO (written after the Colab run).** Report: routing accuracy for each router (with the interval), the
retrieval hit rate for dense vs. re-ranked, the end-to-end plain-code score by question type, the judge
means for correctness, grounding and calibration, and where the judge and the plain-code check disagreed.
Keep the labels honest: the domain-keyword router was written after seeing the eval phrasings, 18 cases
is small, and the judge is the same model family as the generator.

## Part 4: Submit and curate

Checklist (details are in the README and the PR text):

1. Commit this notebook, `data/week8_results.json`, the README and the research note. **Do not commit `data/raw/`.**
2. README for a reader with sixty seconds: what it is, the architecture diagram, the table of which week each component came from, the two or three numbers that matter, and how to run it.
3. Pin the repo on your GitHub profile (or link it prominently from your profile README if you keep the profile private).
4. Read `git log --oneline` for the whole repo: the history should read as Week 1 to Week 8, one coherent story.
5. Link the research note from the README, as in prior weeks.